In [28]:
import cv2
import numpy as np
import pickle
from pathlib import Path

In [29]:
ROI_W = 0.58
ROI_H = 0.58

MIN_AREA = 2500
BLOCK_SIZE = 41   # 21, 31, 41...
C = 3            # 5, 7, 9...
KERNEL_SIZE = 3   # 3 o 5
MATCH_THRESHOLD = 0.18

TEMPLATE_FILE = Path("shape_templates.pkl")

In [30]:
def load_templates():
    if TEMPLATE_FILE.exists():
        with open(TEMPLATE_FILE, "rb") as f:
            return pickle.load(f)
    return {}


def save_templates(templates):
    with open(TEMPLATE_FILE, "wb") as f:
        pickle.dump(templates, f)


def get_center_roi(frame):
    h, w = frame.shape[:2]
    rw = int(w * ROI_W)
    rh = int(h * ROI_H)

    x1 = (w - rw) // 2
    y1 = (h - rh) // 2
    x2 = x1 + rw
    y2 = y1 + rh

    return (x1, y1, x2, y2), frame[y1:y2, x1:x2]


def preprocess_roi(roi):
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 5)

    mask = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        BLOCK_SIZE,
        C
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (KERNEL_SIZE, KERNEL_SIZE)
    )
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    return gray, mask


def get_valid_contours(mask):
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    h, w = mask.shape[:2]
    valid = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < MIN_AREA:
            continue

        x, y, cw, ch = cv2.boundingRect(cnt)

        if x <= 2 or y <= 2 or x + cw >= w - 2 or y + ch >= h - 2:
            continue

        valid.append(cnt)

    valid.sort(key=cv2.contourArea, reverse=True)
    return valid


def contour_info(cnt):
    area = cv2.contourArea(cnt)
    peri = cv2.arcLength(cnt, True)
    x, y, w, h = cv2.boundingRect(cnt)
    aspect = max(w, h) / max(1, min(w, h))

    circularity = 0.0
    if peri > 0:
        circularity = 4.0 * np.pi * area / (peri * peri)

    return {
        "area": area,
        "peri": peri,
        "bbox": (x, y, w, h),
        "aspect": aspect,
        "circularity": circularity,
    }


def classify_shape(cnt, templates):
    if not templates:
        return "no_templates", None

    scores = {}
    for name, tmpl in templates.items():
        score = cv2.matchShapes(cnt, tmpl, cv2.CONTOURS_MATCH_I1, 0.0)
        scores[name] = score

    best_name = min(scores, key=scores.get)
    best_score = scores[best_name]

    if best_score > MATCH_THRESHOLD:
        return "unknown", best_score

    return best_name, best_score

In [31]:
def load_templates():
    if TEMPLATE_FILE.exists():
        with open(TEMPLATE_FILE, "rb") as f:
            return pickle.load(f)
    return {}


def save_templates(templates):
    with open(TEMPLATE_FILE, "wb") as f:
        pickle.dump(templates, f)


def get_center_roi(frame):
    h, w = frame.shape[:2]
    rw = int(w * ROI_W)
    rh = int(h * ROI_H)

    x1 = (w - rw) // 2
    y1 = (h - rh) // 2
    x2 = x1 + rw
    y2 = y1 + rh

    return (x1, y1, x2, y2), frame[y1:y2, x1:x2]


def preprocess_roi(roi):
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 5)

    mask = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        BLOCK_SIZE,
        C
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (KERNEL_SIZE, KERNEL_SIZE)
    )
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    return gray, mask


def get_valid_contours(mask):
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    h, w = mask.shape[:2]
    valid = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < MIN_AREA:
            continue

        x, y, cw, ch = cv2.boundingRect(cnt)

        if x <= 2 or y <= 2 or x + cw >= w - 2 or y + ch >= h - 2:
            continue

        valid.append(cnt)

    valid.sort(key=cv2.contourArea, reverse=True)
    return valid


def contour_info(cnt):
    area = cv2.contourArea(cnt)
    peri = cv2.arcLength(cnt, True)
    x, y, w, h = cv2.boundingRect(cnt)
    aspect = max(w, h) / max(1, min(w, h))

    circularity = 0.0
    if peri > 0:
        circularity = 4.0 * np.pi * area / (peri * peri)

    return {
        "area": area,
        "peri": peri,
        "bbox": (x, y, w, h),
        "aspect": aspect,
        "circularity": circularity,
    }


def classify_shape(cnt, templates):
    if not templates:
        return "no_templates", None

    scores = {}
    for name, tmpl in templates.items():
        score = cv2.matchShapes(cnt, tmpl, cv2.CONTOURS_MATCH_I1, 0.0)
        scores[name] = score

    best_name = min(scores, key=scores.get)
    best_score = scores[best_name]

    if best_score > MATCH_THRESHOLD:
        return "unknown", best_score

    return best_name, best_score

In [32]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

print("camara abierta:", cap.isOpened())

camara abierta: True


In [33]:
templates = load_templates()

print("Teclas:")
print("1 = guardar plantilla circle")
print("2 = guardar plantilla pill")
print("3 = guardar plantilla caplet")
print("s = guardar templates")
print("c = borrar templates en memoria")
print("q = salir")

while True:
    ok, frame = cap.read()
    if not ok:
        print("No se pudo leer frame.")
        break

    display = frame.copy()

    (x1, y1, x2, y2), roi = get_center_roi(frame)
    gray, mask = preprocess_roi(roi)
    contours = get_valid_contours(mask)

    roi_vis = roi.copy()
    cv2.rectangle(display, (x1, y1), (x2, y2), (0, 255, 255), 2)

    biggest = contours[0] if contours else None

    for cnt in contours:
        info = contour_info(cnt)
        x, y, w, h = info["bbox"]

        label, score = classify_shape(cnt, templates)

        cv2.drawContours(roi_vis, [cnt], -1, (0, 255, 0), 2)
        cv2.rectangle(roi_vis, (x, y), (x+w, y+h), (255, 0, 0), 2)

        txt = label
        if score is not None:
            txt += f" {score:.3f}"

        cv2.putText(
            roi_vis, txt, (x, max(20, y-8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2, cv2.LINE_AA
        )

    cv2.imshow("camera", display)
    cv2.imshow("roi", roi_vis)
    cv2.imshow("mask", mask)

    key = cv2.waitKey(1) & 0xFF

    if key == ord("q"):
        break

    elif key == ord("c"):
        templates = {}
        print("Templates borrados en memoria.")

    elif key == ord("s"):
        save_templates(templates)
        print("Templates guardados.")

    elif key in (ord("1"), ord("2"), ord("3")):
        if biggest is None:
            print("No hay contorno valido.")
            continue

        label_map = {
            ord("1"): "circle",
            ord("2"): "pill",
            ord("3"): "caplet",
        }
        label = label_map[key]
        templates[label] = biggest.copy()
        print(f"Template guardado: {label}")

Teclas:
1 = guardar plantilla circle
2 = guardar plantilla pill
3 = guardar plantilla caplet
s = guardar templates
c = borrar templates en memoria
q = salir
Template guardado: circle
Templates guardados.
Templates guardados.


In [34]:
cap.release()
cv2.destroyAllWindows()